In [1]:
!pip install --no-deps /kaggle/input/pymorphy3-whl/pymorphy3-2.0.3-py3-none-any.whl
!pip install --no-deps /kaggle/input/pymorphy3-whl/dawg2_python-0.9.0-py3-none-any.whl
!pip install --no-deps /kaggle/input/pymorphy3-whl/pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl
!pip install --no-deps /kaggle/input/pymorphy3-whl/typing_extensions-4.12.2-py3-none-any.whl


Processing /kaggle/input/pymorphy3-whl/pymorphy3-2.0.3-py3-none-any.whl


Processing /kaggle/input/pymorphy3-whl/dawg2_python-0.9.0-py3-none-any.whl


Processing /kaggle/input/pymorphy3-whl/pymorphy3_dicts_ru-2.4.417150.4580142-py2.py3-none-any.whl


Processing /kaggle/input/pymorphy3-whl/typing_extensions-4.12.2-py3-none-any.whl


typing-extensions is already installed with the same version as the provided wheel. Use --force-reinstall to force an installation of the wheel.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import nltk
import nltk.corpus
import nltk.tokenize
import string
import gensim
from string import punctuation
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import inspect
import pymorphy3
import re
import csv
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import Word2Vec, FastText, KeyedVectors
from tqdm import tqdm


nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Error loading punkt: <urlopen error [Errno -3] Temporary
[nltk_data]     failure in name resolution>


[nltk_data] Error loading stopwords: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


[nltk_data] Error loading punkt_tab: <urlopen error [Errno -3]
[nltk_data]     Temporary failure in name resolution>


False

In [3]:
mapper = {
    'id': 'уникальный номер',
    'tdate': 'дата',
    'tname': 'никнейм',
    'ttext': 'текст',
    'ttype': 'тип',
    'trep': 'количество реплаев',
    'trtw': 'количество ретвитов',
    'tfav': 'количество лайков',
    'tscount': 'количество всех сообщений',
    'tfol': 'количество подписчиков',
    'tfrien': 'количество друзей',
    'listcount': 'количество списков'
}

punctuation += '—'
stopwords = nltk.corpus.stopwords.words('russian')
nescesary_stopwords = ['не', 'нельзя', 'нет', 'ни', 'ничего', 'никогда', 'хорошо', 'можно', 'много', 'лучше', 'конечно', 'даже', 'всегда', 'более', 'больше']

for sw in nescesary_stopwords:
  stopwords.remove(sw)

stopwords.append('всё')

In [4]:
# подгодовка бд
def get_data(path, col_names = mapper.keys()):
  data = pd.read_csv(path, delimiter = ';')
  data.columns = col_names

  return data #sample(lngth)

# предобработка текстовых данных
def text_preprocess(text, morph, stopwords):
  text_no_punct = re.sub('[^А-Яа-яёЁ]', ' ', text)

  text_tokenized = nltk.tokenize.word_tokenize(text_no_punct)
  no_stopwords = [word for word in text_tokenized if word != ' ']
  lemmas = [morph.parse(word)[0].normal_form for word in no_stopwords if word not in stopwords]
  return lemmas

# запись векторов в новый файл
def vectors_to_csv(initial_df, vectors, path):

  df_vectors = pd.DataFrame(tqdm(vectors, desc="Создание DataFrame из векторов", unit="строка"))
  df = pd.concat([initial_df[['id']], df_vectors], axis=1)
  df.to_csv(path, index=False)

  return df

### Получение базы данных

In [5]:
df_neg = get_data(r'/kaggle/input/rutwittcorp/negative.csv')
df_pos = get_data(r'//kaggle/input/rutwittcorp/positive.csv')

df_concat = pd.concat([df_neg, df_pos], ignore_index=True)
df_concat

,id,tdate,tname,ttext,ttype,trep,trtw,tfav,tscount,tfol,tfrien,listcount
0,408906818262687744,1386325957,nugemycejela,"Коллеги сидят рубятся в Urban terror, а я из-з...",-1,0,0,0,26,42,39,0
1,408906858515398656,1386325966,4post21,@elina_4post как говорят обещаного три года жд...,-1,0,0,0,718,49,249,0
2,408906914437685248,1386325980,Poliwake,"Желаю хорошего полёта и удачной посадки,я буду...",-1,0,0,0,10628,207,200,0
3,408906914723295232,1386325980,capyvixowe,"Обновил за каким-то лешим surf, теперь не рабо...",-1,0,0,0,35,17,34,0
4,408906915704737792,1386325980,nunejibaduq,"Котёнка вчера носик разбила, плакала и расстра...",-1,0,0,0,222,62,62,0
...,...,...,...,...,...,...,...,...,...,...,...,...
226827,411368729235054592,1386912922,diminlisenok,"Спала в родительском доме, на своей кровати......",1,0,0,0,1497,56,34,2
226828,411368729424187392,1386912922,qilepocagotu,RT @jebesilofyt: Эх... Мы немного решили сокра...,1,0,1,0,692,225,210,0
226829,411368796537257984,1386912938,DennyChooo,"Что происходит со мной, когда в эфире #proacti...",1,0,0,0,4905,448,193,13
226830,411368797447417856,1386912938,bedowabymir,"""Любимая,я подарю тебе эту звезду..."" Имя како...",1,0,0,0,989,254,251,0


In [6]:
corpus = df_concat['ttext'].to_list()
morph = pymorphy3.MorphAnalyzer()
corpus = [text_preprocess(text, morph, stopwords) for text in corpus]
for sent in corpus:
  for word in sent:
    if word in stopwords:
        sent.remove(word)
max_len = 0


## Извлечение признаков

### LinisCrowd

In [7]:
sent_dict = {}
with open(r'/kaggle/input/word-sentiment/words_all_full_rating.csv', encoding='cp1251') as linis:
  reader = csv.reader(linis, delimiter=';')
  next(reader)
  for row in reader:
    sent_dict[row[0]] = float(row[1].replace(',', ''))

In [8]:
linis_crowd = {'mean': [], 'sum': [], 'max': [], 'min': [], 'positive': [], 'negative': [] }
for text in corpus:
  rates = []
  pos_rates = 0
  neg_rates = 0
  for word in text:
    if word in sent_dict:
      rate = sent_dict[word]
      rates.append(rate)
      if rate > 0:
        pos_rates += 1
      else:
        neg_rates += 1
    else:
      rate = 0
      rates.append(rate)
  if len(rates) == 0:
    linis_crowd['mean'].append(0)
    linis_crowd['sum'].append(0)
    linis_crowd['max'].append(0)
    linis_crowd['min'].append(0)
    linis_crowd['positive'].append(0)
    linis_crowd['negative'].append(0)
  else:
    linis_crowd['mean'].append(sum(rates)/len(rates))
    linis_crowd['sum'].append(sum(rates))
    linis_crowd['max'].append(max(rates))
    linis_crowd['min'].append(min(rates))
    linis_crowd['positive'].append(pos_rates)
    linis_crowd['negative'].append(neg_rates)

In [9]:
linis_crowd_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(linis_crowd)], axis=1 )
linis_crowd_df.to_csv(r'linis_crowd.csv', index=False)
linis_crowd_df

,ttype,mean,sum,max,min,positive,negative
0,-1,-2.857143e-01,-2.000000e+00,0.000000e+00,-2.000000e+00,0,1
1,-1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0,1
2,-1,1.527778e+13,1.222222e+14,1.222222e+14,0.000000e+00,2,1
3,-1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0,1
4,-1,-2.083333e+01,-1.250000e+02,0.000000e+00,-1.250000e+02,0,1
...,...,...,...,...,...,...,...
226827,1,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0,3
226828,1,-1.851852e+13,-2.222222e+14,5.000000e+00,-2.222222e+14,1,2
226829,1,2.666667e+13,1.333333e+14,1.333333e+14,0.000000e+00,1,1
226830,1,1.904762e+13,1.333333e+14,1.333333e+14,0.000000e+00,3,1


### Векторы по частям речи

In [10]:
speech_parts = []

for i, sentence in enumerate(corpus):
    pos_list = [morph.parse(word)[0].tag.POS for word in sentence]
    pos_counts = Counter(pos_list)
    total_words = sum(pos_counts.values())
    relative_freq = {pos: count / total_words for pos, count in pos_counts.items()}

    speech_parts.append({ **relative_freq})

speech_parts_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(speech_parts).fillna(0)], axis=1 )
speech_parts_df.to_csv(r'speech_parts.csv', index=False)
speech_parts_df


,ttype,NOUN,INFN,PRCL,ADJF,ADVB,INTJ,None,PRED,NPRO,CONJ,NUMR,PREP,VERB,ADJS,PRTS,GRND,PRTF,COMP
0,-1,0.428571,0.428571,0.142857,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1,0.250000,0.500000,0.000000,0.250000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,-1,0.250000,0.250000,0.000000,0.250000,0.250000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,-1,0.400000,0.400000,0.200000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-1,0.333333,0.500000,0.000000,0.000000,0.166667,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,0.500000,0.250000,0.000000,0.250000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226828,1,0.333333,0.416667,0.000000,0.083333,0.083333,0.083333,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226829,1,0.400000,0.400000,0.000000,0.200000,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
226830,1,0.571429,0.285714,0.000000,0.142857,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### TF-IDF

In [11]:
vectorizer_tfidf = TfidfVectorizer(min_df=2, max_features=1000)
vectors_tfidf = vectorizer_tfidf.fit_transform(' '.join(sent) for sent in corpus).toarray()

In [12]:
tfidf_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(list(vectors_tfidf))], axis=1 )
tfidf_df.to_csv(r'tfidf.csv', index=False)
tfidf_df

,ttype,0,1,2,3,4,5,6,7,8,...,990,991,992,993,994,995,996,997,998,999
0,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
1,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
2,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
3,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
4,-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
226828,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.327097,0.0,0.0,0.0
226829,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0
226830,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.0


### Wav2Vec

In [13]:
w2vmodel = Word2Vec(corpus, vector_size=300, window=5, min_count=1, workers=4)


In [14]:
w2v = {'text': [], 'vector': []}
for sent in corpus:
  w2v['text'].append(sent)
  vectors = []
  for word in sent:
    vectors.append(w2vmodel.wv[word])
  if len(vectors) == 0:
    w2v['vector'].append(np.zeros(300))
  else:
    w2v['vector'].append(np.array(np.mean(vectors, axis=0)))

In [15]:
w2v_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(w2v['vector'])], axis=1 )
w2v_df.to_csv(r'w2v.csv', index=False)
w2v_df

,ttype,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,0.183009,0.621205,0.032988,0.380573,-0.399954,-0.325540,0.157501,0.852109,0.057216,...,-0.134780,0.456131,0.467896,0.004943,0.161279,0.438377,0.199037,0.086677,0.526828,-0.008486
1,-1,-0.254375,0.794966,0.418410,0.152848,-0.327040,-0.826475,0.181140,1.276808,-0.125573,...,0.322359,0.394496,0.167115,-0.093908,0.381100,0.314614,0.293172,-0.206440,0.629389,-0.216136
2,-1,0.218598,0.582922,0.193415,0.752375,0.046914,-0.746402,0.269834,1.239848,-0.191107,...,0.049869,0.500371,0.170719,0.132367,0.329075,0.480938,0.293632,0.052636,-0.062641,-0.174618
3,-1,-0.129511,0.419715,0.124214,0.323723,-0.394384,-0.268023,-0.016591,0.815302,0.059434,...,0.040759,0.309255,0.415526,-0.062462,0.110162,0.228017,0.157463,0.038667,0.521196,-0.035253
4,-1,0.004577,0.491403,0.102976,0.345904,-0.089652,-0.573084,0.286073,0.691324,-0.021673,...,0.009241,0.437608,0.236276,0.108071,0.183847,0.396574,0.190087,-0.030411,0.164304,-0.127712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,-0.020314,1.013968,0.071345,0.837819,-0.464706,-0.442750,0.414294,0.900957,0.118230,...,-0.090190,0.608508,0.231338,0.329453,-0.003248,0.368921,0.270564,0.257675,0.212510,0.040794
226828,1,-0.011750,0.520024,0.074517,0.425495,-0.281394,-0.318537,0.279072,0.810421,0.127874,...,-0.074341,0.545471,0.404012,0.186695,0.226597,0.397933,0.180905,0.028205,0.382510,-0.084377
226829,1,-0.042748,0.654329,0.074157,0.597583,-0.075568,-0.765072,0.240880,1.284110,-0.101997,...,0.245145,0.344289,0.201723,0.011147,0.426342,0.792089,0.019886,-0.098199,0.291323,-0.155936
226830,1,0.019838,0.413789,0.087393,0.242063,0.016326,-0.638216,0.033276,0.887872,0.048800,...,0.208247,0.202287,0.378399,0.183365,0.365487,0.512832,-0.008025,-0.341651,0.521332,-0.315595


In [16]:
model_path = "ruwikiruscorpora_upos_cbow_300_10_2021.bin.gz"
w2vmodel_pretrained = gensim.models.KeyedVectors.load_word2vec_format('/kaggle/input/w2v/pytorch/default/1/220/model.bin', encoding='utf-8', unicode_errors='ignore', binary=True)
w2vmodel_pretrained.most_similar('кот_NOUN')

[('кошка_NOUN', 0.7819064259529114),
 ('кот_PROPN', 0.7419374585151672),
 ('пес_NOUN', 0.7320025563240051),
 ('котенок_NOUN', 0.7187714576721191),
 ('мышонок_NOUN', 0.693092942237854),
 ('котёнка_NOUN', 0.688942015171051),
 ('щенок_NOUN', 0.6635671257972717),
 ('собачка_NOUN', 0.6484753489494324),
 ('шарик_PROPN', 0.6482844352722168),
 ('кролик_NOUN', 0.6440559029579163)]

In [17]:
w2vmodel_cleaned = {key.split('_')[0]: w2vmodel_pretrained[key] for key in w2vmodel_pretrained.key_to_index}

In [18]:
p = morph.parse('стали')[0]
first_tag_part = p.tag.POS
first_tag_part

'VERB'

In [19]:
w2v_pretrained = {'text': [], 'vector': []}
for sent in corpus:
  w2v_pretrained['text'].append(sent)
  vectors = []
  for word in sent:
    try:
      #first_tag_part = morph.parse(word)[0].tag.POS
      vectors.append(w2vmodel_cleaned[word])
    except KeyError:
      vectors.append(np.zeros(300))
  if len(vectors) == 0:
    w2v_pretrained['vector'].append(np.zeros(300))
  else:
    w2v_pretrained['vector'].append(np.array(np.mean(vectors, axis=0)))


In [20]:
w2v_pretrained_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(w2v_pretrained['vector'])], axis=1 )
w2v_pretrained_df.to_csv(r'w2v_pretrained.csv', index=False)
w2v_pretrained_df

,ttype,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,-0.029762,0.140600,0.133712,0.035470,0.075019,0.000115,0.020909,-0.183372,-0.129579,...,0.001369,-0.056840,0.095804,0.272807,-0.103780,-0.115116,0.066937,-0.071950,-0.190474,0.074695
1,-1,0.306034,-0.049628,0.511388,-0.133388,0.627173,0.016263,0.544889,-0.487313,0.000532,...,0.036866,-0.223869,-0.066166,0.483355,0.093465,-0.020690,0.107873,-0.225651,-0.253817,0.056730
2,-1,0.267032,-0.640748,0.582422,0.283539,-0.174290,-0.477183,0.321560,0.256583,-0.281591,...,0.357824,0.087153,-0.111161,0.850645,-0.210355,-0.024791,0.109316,0.117970,-0.081024,-0.047721
3,-1,-0.507809,-0.426009,-0.312520,0.239345,-0.011074,-0.392596,-0.153868,0.398518,0.048894,...,-0.370990,0.066085,-0.149617,-0.014163,0.191351,0.530103,-0.270313,0.322614,-0.168994,-0.810987
4,-1,-0.151107,0.051494,0.031412,0.336166,0.267406,-0.035506,0.247768,0.269623,-0.082467,...,-0.319396,0.049476,0.367956,0.509886,-0.392611,-0.051570,0.082933,-0.039620,-0.273361,0.083118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,-0.825148,-0.744613,-0.097037,-0.130733,0.342291,-0.312562,-0.105434,0.087703,0.185391,...,0.056417,0.213189,0.015083,0.104531,0.737710,0.004642,0.449725,0.146682,-1.306366,0.327593
226828,1,-0.040685,0.543406,0.377491,0.158974,0.284649,0.254869,0.498592,-0.407123,-0.465529,...,0.058191,0.089812,-0.221591,-0.894840,-0.096231,0.406389,-0.168242,0.338954,-0.175192,0.738967
226829,1,0.637349,-0.791408,0.590899,0.196915,0.144282,-0.556750,0.659850,0.167698,1.330819,...,-0.230145,0.133529,-0.669698,0.561082,0.062336,0.171485,0.006728,-0.751613,0.143604,-0.085217
226830,1,0.123073,-0.078700,-0.143195,0.219847,0.025707,-0.316972,-0.290462,0.812479,0.504588,...,-0.709260,0.251382,0.012610,0.319612,-0.208570,-0.015270,0.315885,-0.595139,-0.548554,0.022259


### FastText

In [21]:
ftmodel = FastText(corpus, vector_size=300, window=5, min_count=5)
ftmodel.wv.most_similar('хорошо')

[('хорошоо', 0.9878456592559814),
 ('хорошооо', 0.9833037257194519),
 ('хорошоя', 0.9806843400001526),
 ('нехорошо', 0.9460980892181396),
 ('хорошой', 0.9017946720123291),
 ('хорошенько', 0.8891957998275757),
 ('хоро', 0.8803530931472778),
 ('плохо', 0.8607558012008667),
 ('плохоо', 0.8606411218643188),
 ('хором', 0.8506401181221008)]

In [22]:
fasttext = {'text': [], 'vector': []}
for sent in corpus:
  fasttext['text'].append(sent)
  vectors = []
  for word in sent:
    vectors.append(ftmodel.wv[word])
  if len(vectors) == 0:
    fasttext['vector'].append(np.zeros(300))
  else:
    fasttext['vector'].append(np.array(np.mean(vectors, axis=0)))


In [23]:
fasttext_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(fasttext['vector'])], axis=1 )
fasttext_df.to_csv(r'fasttext.csv', index=False)
fasttext_df

,ttype,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,0.874088,0.207821,-1.026979,-0.319311,-0.752779,0.470951,0.826582,-0.553837,-0.147888,...,-0.118886,-0.078070,0.152340,-0.028215,0.334890,0.159489,-0.263139,0.207233,0.124690,0.495681
1,-1,-0.749341,0.245129,0.783753,-0.537444,0.169011,-0.339662,-0.216578,0.195465,0.151385,...,-0.064373,-0.260368,-0.033377,-0.092747,-0.464812,-0.040437,-0.625073,0.044970,-0.436060,-0.400707
2,-1,-0.006093,0.245561,0.062901,-0.270355,0.188685,-0.162870,0.166524,0.025708,0.145003,...,0.363567,0.084797,-0.568130,0.209969,-0.405758,-0.280179,-0.015455,-0.030927,-0.140277,0.545736
3,-1,0.615189,0.319120,-0.430734,-0.309640,-0.624744,0.286154,1.052730,-0.609741,-0.051070,...,0.057718,-0.084436,0.294805,-0.060031,0.090034,-0.330960,-0.354715,0.223202,0.458536,0.667943
4,-1,0.127984,0.377183,-0.362585,-0.441668,0.047590,-0.076376,0.400117,-0.255323,-0.061056,...,0.000081,-0.159146,-0.480233,0.043844,-0.147349,0.177791,-0.004952,-0.144688,-0.156435,0.169599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,0.087554,0.389953,-0.459682,-0.481419,-0.006515,-0.190970,0.552722,-0.546561,-0.049803,...,-0.038620,-0.085507,-0.301950,0.138054,-0.414500,0.426454,-0.054135,-0.075249,-0.199635,0.298133
226828,1,0.191460,0.333021,-0.350870,-0.411993,-0.106024,0.078574,0.481681,-0.248023,-0.002672,...,0.046940,-0.017898,-0.047422,0.056644,-0.152605,0.139488,-0.040052,-0.143375,-0.236698,0.138931
226829,1,-0.198479,0.310311,-0.084504,-0.361827,0.355058,-0.186478,-0.292518,0.436804,0.050926,...,0.246507,-0.189859,-0.631880,0.218791,-0.131038,-0.090591,0.157252,0.018591,0.009489,-0.167263
226830,1,-0.115208,0.233455,0.147280,-0.332559,0.343725,-0.397851,0.128138,0.245717,-0.105110,...,-0.134027,-0.217908,-0.302395,0.097315,-0.235292,-0.114772,0.125028,-0.109527,0.231031,0.081252


In [24]:
ftmodel_pretrained = gensim.models.KeyedVectors.load('/kaggle/input/fasttext/pytorch/default/1/213/model.model')
ftmodel_pretrained.most_similar('кот')

[('кошка', 0.8589062094688416),
 ('кота', 0.8251551985740662),
 ('котенок', 0.7663764357566833),
 ('собака', 0.7647724151611328),
 ('пес', 0.7552706599235535),
 ('котик', 0.7500638961791992),
 ('котенка', 0.738452136516571),
 ('песик', 0.6929449439048767),
 ('кошак', 0.688758373260498),
 ('песика', 0.687090277671814)]

In [25]:
ft_pretrained = {'text': [], 'vector': []}
for sent in corpus:
  ft_pretrained['text'].append(sent)
  vectors = []
  for word in sent:
    vectors.append(ftmodel_pretrained[word])
  if len(vectors) == 0:
    ft_pretrained['vector'].append(np.zeros(300))
  else:
    ft_pretrained['vector'].append(np.array(np.mean(vectors, axis=0)))

In [26]:
fasttext_pretrainef_df = pd.concat([df_concat[['ttype']],  pd.DataFrame(ft_pretrained['vector'])], axis=1 )
fasttext_pretrainef_df.to_csv(r'fasttext_pretr.csv', index=False)
fasttext_pretrainef_df

,ttype,0,1,2,3,4,5,6,7,8,...,290,291,292,293,294,295,296,297,298,299
0,-1,0.122010,0.077201,-0.020517,0.172850,0.070921,-0.009873,0.103921,-0.023100,0.101336,...,0.203994,0.113248,0.111351,0.021807,-0.182379,0.034013,0.046017,-0.062362,-0.182148,-0.060355
1,-1,-0.063084,-0.015199,-0.170328,0.011000,0.212626,0.262083,-0.008568,0.071123,-0.030609,...,0.061179,-0.031331,0.097060,0.142485,-0.079269,-0.090395,0.149548,0.015660,-0.199339,0.008175
2,-1,-0.019249,0.042326,-0.079763,0.447684,0.219215,0.058586,-0.009051,-0.075183,0.282244,...,-0.002557,0.090420,0.178318,0.071817,-0.141936,0.025786,0.111765,-0.091589,-0.100430,0.130720
3,-1,0.049566,0.116262,-0.100898,0.108603,0.131318,0.037689,0.003054,0.046993,0.015449,...,0.091523,0.156664,0.128601,0.177975,-0.135454,-0.011906,0.170782,-0.173338,-0.188826,-0.048318
4,-1,0.024385,0.025933,-0.117150,0.043527,0.084858,0.011253,0.080193,-0.025402,-0.068903,...,-0.009304,0.163140,0.113939,0.101865,-0.172338,0.041469,0.080220,-0.146384,-0.057868,-0.086599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
226827,1,0.063053,0.068928,-0.038545,0.062647,0.004820,0.013285,-0.021656,-0.076963,-0.026101,...,0.001430,0.073476,0.090030,0.110430,-0.095829,-0.042353,0.067769,-0.106041,-0.222030,-0.058689
226828,1,0.041691,-0.022612,-0.079342,0.170320,0.066096,-0.069708,0.080173,0.060388,0.070476,...,0.016562,0.127788,0.085547,0.052909,-0.036571,0.078502,0.169290,-0.105821,-0.038792,0.004138
226829,1,0.199276,0.098200,-0.210514,0.024557,0.264724,0.048264,-0.107764,-0.244315,-0.088067,...,-0.071458,0.141709,0.118787,0.074310,-0.140691,-0.041034,0.137952,0.027621,0.017861,-0.034694
226830,1,-0.010569,0.066078,-0.099928,0.149968,0.136639,0.188381,-0.022298,-0.221408,0.019148,...,-0.067583,0.190938,0.247615,-0.003015,-0.181508,-0.001297,-0.098913,0.015980,-0.179948,0.087572


## Обучение

In [27]:
results_lr = {}

results_svc = {}

def get_vectors(path):
  data = pd.read_csv(path)
  features = data.drop('ttype', axis=1)
  labels = data['ttype']
  return features, labels

def train(features, lables, model, data_type):
  X_train, X_test, y_train, y_test = train_test_split(features, lables, test_size=0.2, random_state=42)
  model.fit(X_train, y_train)
  y_pred = model.predict(X_test)
  if model == lr_model:
    results_lr[data_type] = accuracy_score(y_test, y_pred)
  if model == svc_model:
    results_svc[data_type] = accuracy_score(y_test, y_pred)
  print(classification_report(y_test, y_pred))


In [28]:
lr_model = LogisticRegression()
svc_model = SVC()

In [29]:
linis_crowd_features, linis_crowd_labels = get_vectors(r'/kaggle/working/linis_crowd.csv')
train(linis_crowd_features, linis_crowd_labels, lr_model, 'linis crowd')
train(linis_crowd_features, linis_crowd_labels, svc_model, 'linis crowd')

              precision    recall  f1-score   support

          -1       0.53      0.76      0.63     22286
           1       0.61      0.36      0.45     23081

    accuracy                           0.56     45367
   macro avg       0.57      0.56      0.54     45367
weighted avg       0.57      0.56      0.54     45367



              precision    recall  f1-score   support

          -1       0.53      0.79      0.63     22286
           1       0.61      0.32      0.42     23081

    accuracy                           0.55     45367
   macro avg       0.57      0.55      0.52     45367
weighted avg       0.57      0.55      0.52     45367



In [30]:
speech_parts_features, speech_parts_labels = get_vectors(r'/kaggle/working/speech_parts.csv')
train(speech_parts_features, speech_parts_labels, lr_model, 'speech parts')
train(speech_parts_features, speech_parts_labels, svc_model, 'speech parts')

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_logistic.py:458: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              precision    recall  f1-score   support

          -1       0.57      0.49      0.53     22286
           1       0.57      0.65      0.61     23081

    accuracy                           0.57     45367
   macro avg       0.57      0.57      0.57     45367
weighted avg       0.57      0.57      0.57     45367



              precision    recall  f1-score   support

          -1       0.57      0.54      0.55     22286
           1       0.58      0.60      0.59     23081

    accuracy                           0.57     45367
   macro avg       0.57      0.57      0.57     45367
weighted avg       0.57      0.57      0.57     45367



In [31]:
tfidf_features, tfidf_labels = get_vectors(r'/kaggle/working/tfidf.csv')
train(tfidf_features, tfidf_labels, lr_model, 'tfidf')
train(tfidf_features, tfidf_labels, svc_model, 'tfidf')

              precision    recall  f1-score   support

          -1       0.70      0.64      0.67     22286
           1       0.68      0.73      0.71     23081

    accuracy                           0.69     45367
   macro avg       0.69      0.69      0.69     45367
weighted avg       0.69      0.69      0.69     45367



In [ ]:
wav2vec_features, wav2vec_labels = get_vectors(r'/kaggle/working/w2v.csv')
train(wav2vec_features, wav2vec_labels, lr_model, 'wav2vec')
train(wav2vec_features, wav2vec_labels, svc_model, 'wav2vec')

In [ ]:
wav2vec_pretr_features, wav2vec_pretr_labels = get_vectors(r'/kaggle/working/w2v_pretrained.csv')
train(wav2vec_pretr_features, wav2vec_pretr_labels, lr_model, 'wav2vec_pretrained')
train(wav2vec_pretr_features, wav2vec_pretr_labels, svc_model, 'wav2vec_pretrained')

In [ ]:
ft_features, ft_labels= get_vectors(r'/kaggle/working/fasttext.csv')
train(ft_features, ft_labels, lr_model, 'fast text')
train(ft_features, ft_labels, svc_model, 'fast text')

In [ ]:
ft_pretr_features, ft_pretr_labels  = get_vectors(r'/kaggle/working/fasttext_pretr.csv')
train(ft_pretr_features, ft_pretr_labels, lr_model, 'fast text pretrained')
train(ft_pretr_features, ft_pretr_labels , svc_model, 'fast text pretrained')

### Результаты классификации с помщью логистической регрессии

In [ ]:
results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_lr_df

In [ ]:
results_lr_df = pd.DataFrame.from_dict(results_lr, orient='index', columns=['Accuracy'])
results_lr_df

### Результаты классификации с помщью метода опорных векторов

In [ ]:
results_svc_df = pd.DataFrame.from_dict(results_svc, orient='index', columns=['Accuracy'])
results_svc_df

In [ ]:
results_svc_df = pd.DataFrame.from_dict(results_svc, orient='index', columns=['Accuracy'])
results_svc_df